# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [2]:
# 상위 폴더 path 설정
import sys
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from common_config import llm_connect

temperature = 0
max_tokens = 2086

llm = llm_connect()

## 1. PromptTemplate - 단일 문자열 템플릿

In [3]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [5]:
from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(overfitting)은 **학습 데이터에는 너무 잘 맞지만, 새로운 데이터에는 성능이 떨어지는 현상**입니다.

쉽게 말해:
- 시험 문제를 **기억**만 해서
- 비슷한 기출문제는 잘 풀지만
- 처음 보는 문제는 잘 못 푸는 상태예요.

### 왜 생기나요?
보통 모델이:
- 너무 복잡하거나
- 학습 데이터를 너무 오래 학습했거나
- 데이터가 너무 적거나
- 잡음까지 같이 배워버릴 때 생깁니다.

### 특징
- **훈련 성능은 높음**
- **검증/테스트 성능은 낮음**

예:
- 훈련 정확도 99%
- 테스트 정확도 70%
→ 이런 경우 과적합을 의심할 수 있습니다.

### 줄이는 방법
- 더 많은 데이터 사용
- 모델 단순화
- 정규화(L1, L2)
- 드롭아웃(dropout)
- 조기 종료(early stopping)
- 데이터 증강(data augmentation)

원하시면 제가 **과적합 vs 일반화**도 같이 비교해서 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [7]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

In [8]:
messages

ChatPromptValue(messages=[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={})])

In [9]:
response = llm.invoke(messages)
print(response.content)

당신의 이름은 홍길동입니다.


In [10]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='당신의 이름은 홍길동입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [11]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [13]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "오늘 날씨가 정말 너무 추워요"
})

response = llm.invoke(messages)
print(response.content)

한국어: 오늘 날씨가 정말 너무 추워요.  
영어: The weather is really, really cold today.
